In [1]:
import pandas as pd

# Load the Report_KPI Mapping sheet
df = pd.read_excel("Copy of KPI Mapping.xlsx", sheet_name="Report_KPI Mapping")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()


Shape: (145931, 14)
Columns: ['id', 'File Type', 'Domain', 'Key', 'Workspace', 'Report Name', 'Table Name', 'KPI/Used Columns', 'Type', 'Expression', 'Repeated', 'Consolidate (Yes/No)', 'Consolidated Report Name', 'Table Type']


,id,File Type,Domain,Key,Workspace,Report Name,Table Name,KPI/Used Columns,Type,Expression,Repeated,Consolidate (Yes/No),Consolidated Report Name,Table Type
0,1,PBI,NaN,2.\tService__,2.\tService,NaN,NaN,NaN,NaN,NaN,Yes,Yes,Test,Dimension table
1,2,PBI,NaN,2.\tService__,2.\tService,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dimension table
2,3,PBI,NaN,2.\tService__,2.\tService,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dimension table
3,4,PBI,NaN,2.\tService__,2.\tService,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dimension table
4,5,PBI,NaN,2.\tService__,2.\tService,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dimension table


In [2]:
# Make a copy of the dataframe in memory
df_copy = df.copy()
print(f"df_copy created (backup in memory): {df_copy.shape}")

# Remove rows where only Report Name is missing (other key fields are present)
key_cols = [c for c in df.columns if c != 'Report Name']
mask = df['Report Name'].isnull() & ~df[key_cols].isnull().all(axis=1)
print(f"\nRows where only Report Name is missing: {mask.sum()}")

df = df[~mask].copy()
print(f"Cleaned df shape: {df.shape}")
print(f"Rows removed: {len(df_copy) - len(df)}")

df_copy created (backup in memory): (145931, 14)

Rows where only Report Name is missing: 1767
Cleaned df shape: (144164, 14)
Rows removed: 1767


In [3]:
# Inspect the Type column to understand KPI buckets
print("Type value counts (raw):")
print(df['Type'].value_counts(dropna=False))
print(f"\nUnique non-null Types: {df['Type'].dropna().unique()}")
print(f"\nExpression missing count: {df['Expression'].isnull().sum()} of {len(df)}")
print(f"KPI/Used Columns missing count: {df['KPI/Used Columns'].isnull().sum()} of {len(df)}")


Type value counts (raw):
Type
Column         107173
Measure         22039
Cal. Column     12389
NaN              2563
Name: count, dtype: int64

Unique non-null Types: <StringArray>
['Column', 'Measure', 'Cal. Column']
Length: 3, dtype: str

Expression missing count: 109736 of 144164
KPI/Used Columns missing count: 2564 of 144164


## Step 1 — Bucket KPIs into the 3 Types

Bucket rows into:
- **Column** — similarity on KPI name only, threshold **90%**
- **Measure** — similarity on KPI name **AND** expression, threshold **90%**
- **Cal. Column** — similarity on KPI name **AND** expression, threshold **90%**

Rows with missing KPI/Used Columns name are dropped (cannot consolidate what has no name).


In [4]:
# Drop rows missing the KPI name (cannot consolidate something with no identifier)
before = len(df)
df_kpi = df[df['KPI/Used Columns'].notna() & df['Type'].notna()].copy()
print(f"Dropped {before - len(df_kpi)} rows missing KPI name or Type")
print(f"Working set: {len(df_kpi)} rows\n")

# Split into the three buckets
bucket_column   = df_kpi[df_kpi['Type'] == 'Column'].copy()
bucket_measure  = df_kpi[df_kpi['Type'] == 'Measure'].copy()
bucket_calcol   = df_kpi[df_kpi['Type'] == 'Cal. Column'].copy()

print(f"Bucket  Column      : {len(bucket_column):>7,} rows  |  unique names: {bucket_column['KPI/Used Columns'].nunique():>6,}")
print(f"Bucket  Measure     : {len(bucket_measure):>7,} rows  |  unique names: {bucket_measure['KPI/Used Columns'].nunique():>6,}")
print(f"Bucket  Cal. Column : {len(bucket_calcol):>7,} rows  |  unique names: {bucket_calcol['KPI/Used Columns'].nunique():>6,}")


Dropped 2564 rows missing KPI name or Type
Working set: 141600 rows

Bucket  Column      : 107,173 rows  |  unique names: 14,190
Bucket  Measure     :  22,039 rows  |  unique names:  5,740
Bucket  Cal. Column :  12,388 rows  |  unique names:  3,503


## Step 2 — Similarity Config & Normalizers

Single uniform threshold.

| Setting | Value | Rationale |
|---|---|---|
| `SIMILARITY_THRESHOLD` | **90** | for Columns; same bar applied uniformly |

**Matching rule per bucket:**
- **Column** → `fuzz(name_a, name_b) ≥ 90`
- **Measure** → `fuzz("name_a | expr_a", "name_b | expr_b") ≥ 90` — name and expression concatenated, single fuzzy match
- **Cal. Column** → same concatenated rule as Measure

"Consider's both name AND expression" without introducing any weights or blending math. The combined string is what gets matched.

**Normalization:**
- Lowercase, strip, collapse whitespace
- For Expressions: also flatten DAX line breaks/tabs so `SUM( Sales[Amt] )` matches `SUM(Sales[Amt])`


In [ ]:
import re
from rapidfuzz import fuzz, process

# ---- CONFIG ----------------------------------------------------------------
SIMILARITY_THRESHOLD = 90    #applied uniformly to all 3 buckets

# ---- NORMALIZERS -----------------------------------------------------------
_ws_re = re.compile(r"\s+")

def norm_name(s) -> str:
    """Lowercase, strip, collapse whitespace. Used for KPI names."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    return _ws_re.sub(" ", str(s).strip().lower())

def norm_expr(s) -> str:
    """Same as norm_name but also flattens DAX line breaks/tabs."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    return _ws_re.sub(" ", str(s).replace("\n", " ").replace("\r", " ").replace("\t", " ").strip().lower())

def make_name_expr_key(name: str, expr: str) -> str:
    """Concatenate normalized name + expression into a single string for fuzzy matching.
    The separator ' || ' is unlikely to appear in real KPI text."""
    return f"{name} || {expr}"

# Apply normalization to each bucket (in-place on the bucket DataFrames)
for b in (bucket_column, bucket_measure, bucket_calcol):
    b['_name_norm'] = b['KPI/Used Columns'].map(norm_name)

for b in (bucket_measure, bucket_calcol):
    b['_expr_norm']     = b['Expression'].map(norm_expr)
    b['_name_expr_key'] = [make_name_expr_key(n, e) for n, e in zip(b['_name_norm'], b['_expr_norm'])]

print("Normalization complete.")
print(f"  Column      : {bucket_column['_name_norm'].nunique():>6,} unique normalized names")
print(f"  Measure     : {bucket_measure['_name_norm'].nunique():>6,} unique normalized names | "
      f"{bucket_measure['_name_expr_key'].nunique():>6,} unique (name|expr) keys")
print(f"  Cal. Column : {bucket_calcol['_name_norm'].nunique():>6,} unique normalized names | "
      f"{bucket_calcol['_name_expr_key'].nunique():>6,} unique (name|expr) keys")


Normalization complete.
  Column      : 14,186 unique normalized names
  Measure     :  5,740 unique normalized names |  6,304 unique (name|expr) keys
  Cal. Column :  3,503 unique normalized names |  4,552 unique (name|expr) keys


## Step 3 — Clustering Engine

Single clustering function backed by Union-Find:

`cluster_by_strings(strings, threshold)` — runs pairwise `token_sort_ratio` on a list of normalized strings, unions any pair scoring ≥ threshold.

Usage:
- **Column** bucket → pass normalized names
- **Measure / Cal. Column** buckets → pass concatenated `"name || expression"` keys

No weights, no blending — one rule, one threshold, applied uniformly. Backed by `rapidfuzz.process.cdist` (parallel C++).


In [14]:
import numpy as np

class UnionFind:
    """Minimal union-find with path compression."""
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb

def cluster_by_strings(unique_strings, threshold):
    """Cluster a list of unique strings by fuzzy similarity.
    Returns: dict {string -> cluster_root_index}
    """
    n = len(unique_strings)
    if n == 0:
        return {}
    scores = process.cdist(
        unique_strings, unique_strings,
        scorer=fuzz.token_sort_ratio,
        dtype=np.uint8,
        workers=-1,
    )
    ii, jj = np.where(scores >= threshold)
    mask = ii < jj
    ii, jj = ii[mask], jj[mask]
    uf = UnionFind(n)
    for i, j in zip(ii, jj):
        uf.union(int(i), int(j))
    return {unique_strings[i]: uf.find(i) for i in range(n)}

print("Clustering engine ready.")


Clustering engine ready.


## Step 4 — Run Clustering on Each Bucket

Each bucket gets a `_cluster_id` that's **unique across all buckets** (prefixed with `COL-`, `MEAS-`, `CALC-`).
After this step every row in `df_kpi` knows which consolidation cluster it belongs to.


In [26]:
import time

# ---- Column bucket: match on normalized name ------------------------------
t0 = time.time()
col_unique_names = sorted(bucket_column['_name_norm'].dropna().unique().tolist())
col_clusters = cluster_by_strings(col_unique_names, SIMILARITY_THRESHOLD)
bucket_column['_cluster_raw'] = bucket_column['_name_norm'].map(col_clusters)
print(f"Column      : clustered {len(col_unique_names):,} unique names in {time.time()-t0:.1f}s")

# ---- Measure bucket: match on concatenated 'name || expression' -----------
t0 = time.time()
meas_keys = sorted(bucket_measure['_name_expr_key'].dropna().unique().tolist())
meas_clusters = cluster_by_strings(meas_keys, SIMILARITY_THRESHOLD)
bucket_measure['_cluster_raw'] = bucket_measure['_name_expr_key'].map(meas_clusters)
print(f"Measure     : clustered {len(meas_keys):,} unique (name|expr) keys in {time.time()-t0:.1f}s")

# ---- Cal. Column bucket: match on concatenated 'name || expression' -------
t0 = time.time()
calc_keys = sorted(bucket_calcol['_name_expr_key'].dropna().unique().tolist())
calc_clusters = cluster_by_strings(calc_keys, SIMILARITY_THRESHOLD)
bucket_calcol['_cluster_raw'] = bucket_calcol['_name_expr_key'].map(calc_clusters)
print(f"Cal. Column : clustered {len(calc_keys):,} unique (name|expr) keys in {time.time()-t0:.1f}s")

# ---- Finalize: drop singletons + renumber real clusters 1..N by impact ----
# Singletons (clusters with only one distinct variant) carry no consolidation
# value, so they get an EMPTY cluster_id ('') and are excluded from all cluster
# sheets. Real clusters are renumbered 1..N within each bucket, ordered by
# impact (row count desc) — so COL-1 is always the biggest-impact Column
# cluster, MEAS-1 the biggest Measure cluster, etc. Raw Union-Find root indices
# (which can be in the 10K range because they're positions in the original
# sorted unique-names list) are discarded.
def _finalize_clusters(bucket_df: pd.DataFrame, variant_col: str, prefix: str) -> int:
    variant_counts = bucket_df.groupby('_cluster_raw')[variant_col].nunique()
    singleton_raw = set(variant_counts[variant_counts == 1].index)
    real = bucket_df[~bucket_df['_cluster_raw'].isin(singleton_raw)]
    impact = real.groupby('_cluster_raw').size().sort_values(ascending=False, kind='stable')
    new_ids = {raw: f'{prefix}{i+1}' for i, raw in enumerate(impact.index)}
    bucket_df['_cluster_id'] = bucket_df['_cluster_raw'].map(
        lambda raw: '' if (pd.isna(raw) or raw in singleton_raw) else new_ids[raw]
    )
    return len(new_ids)

n_col_clusters  = _finalize_clusters(bucket_column,  '_name_norm',     'COL-')
n_meas_clusters = _finalize_clusters(bucket_measure, '_name_expr_key', 'MEAS-')
n_calc_clusters = _finalize_clusters(bucket_calcol,  '_name_expr_key', 'CALC-')

print(f"\nReal clusters (singletons dropped; renumbered 1..N by impact):")
print(f"  Column      : {n_col_clusters:>5,} real clusters  +  {(bucket_column ['_cluster_id']=='').sum():>6,} unclustered rows")
print(f"  Measure     : {n_meas_clusters:>5,} real clusters  +  {(bucket_measure['_cluster_id']=='').sum():>6,} unclustered rows")
print(f"  Cal. Column : {n_calc_clusters:>5,} real clusters  +  {(bucket_calcol ['_cluster_id']=='').sum():>6,} unclustered rows")

# ---- Combined view --------------------------------------------------------
df_clustered = pd.concat([bucket_column, bucket_measure, bucket_calcol], ignore_index=True)
print(f"\nTotal rows: {len(df_clustered):,}")


Column      : clustered 14,186 unique names in 29.9s
Measure     : clustered 6,304 unique (name|expr) keys in 50.7s
Cal. Column : clustered 4,552 unique (name|expr) keys in 44.1s

Real clusters (singletons dropped; renumbered 1..N by impact):
  Column      : 1,397 real clusters  +  71,465 unclustered rows
  Measure     :   938 real clusters  +  10,511 unclustered rows
  Cal. Column :   471 real clusters  +   7,990 unclustered rows

Total rows: 141,600


## Step 5 — Consolidation Summary

A cluster is a **consolidation candidate** when it contains **2 or more distinct KPI variants**. Singletons were already collapsed into `cluster_id = '-1'` in Step 4 and are excluded here.

Two outputs are built:

**A. Wide summary** (`candidates_*`) — one row per cluster:
- `canonical_kpi` (the merge target) + `canonical_selection_reason`
- `variant_count`, `total_occurrences`, `reports_affected_count`
- Pipe-delimited `variants` and `reports_affected`
- For Measure / Cal.Col: `canonical_expression`, `variant_expressions`, `distinct_expr_count`

**B. Long-format per-member** (`clusters_long`) — one row per **variant** in a cluster (built in the next cell). This mirrors the format of the CRED Scoring Engine's `Consolidation Clusters` sheet so the two outputs feel consistent.


In [30]:
EXCEL_CELL_LIMIT = 32767   # Excel hard limit per cell
_EXPR_SAMPLE_COUNT  = 3    # how many variant expressions to embed in the summary
_EXPR_MAX_CHARS     = 8000 # per-expression cap

def _pick_canonical_variant(grp: pd.DataFrame, has_expr: bool) -> tuple[str, str]:
    """Pick the canonical (name, expression) for a cluster.

    Rule: choose the most-frequent VARIANT (i.e. the most-frequent name for Column,
    or the most-frequent (name, expression) tuple for Measure / Cal.Col). Ties are
    broken by shortest then alphabetical on the name. This guarantees that exactly
    ONE variant in the cluster matches both canonical_kpi and canonical_expression.
    """
    names = grp['KPI/Used Columns'].astype(str)
    if has_expr:
        exprs = grp['Expression'].fillna('').astype(str)
        pairs = list(zip(names, exprs))
    else:
        pairs = [(n, '') for n in names]

    pair_counts = pd.Series(pairs).value_counts()
    top_count = pair_counts.max()
    tied = pair_counts[pair_counts == top_count].index.tolist()
    # Tiebreak: shortest name, then alphabetical name, then shortest expr
    tied_sorted = sorted(tied, key=lambda p: (len(p[0]), p[0], len(p[1]), p[1]))
    return tied_sorted[0]  # (canon_name, canon_expr)

def _truncate_for_excel(s: str, limit: int = EXCEL_CELL_LIMIT) -> str:
    """Truncate with a marker so it's obvious — never silently lose data."""
    if s is None:
        return ''
    if len(s) <= limit:
        return s
    marker = f"…[truncated {len(s) - limit + 20} chars]"
    return s[: limit - len(marker)] + marker

def build_cluster_summary(bucket_df: pd.DataFrame, kpi_type: str, has_expr: bool) -> pd.DataFrame:
    """Produce one summary row per real cluster (singletons with empty cluster_id are excluded)."""
    real = bucket_df[bucket_df['_cluster_id'] != '']
    rows = []
    for cid, grp in real.groupby('_cluster_id', sort=False):
        canon_name, canon_expr = _pick_canonical_variant(grp, has_expr)

        names = grp['KPI/Used Columns'].astype(str)
        if has_expr:
            variant_keys = grp[['KPI/Used Columns', 'Expression']].astype(str).agg(' || '.join, axis=1)
        else:
            variant_keys = names

        variants = sorted(names.unique().tolist())
        reports_affected = sorted(grp['Report Name'].dropna().astype(str).unique().tolist())

        row = {
            'cluster_id': cid,
            'kpi_type': kpi_type,
            'canonical_kpi': canon_name,
            'variant_count': variant_keys.nunique(),
            'total_occurrences': len(grp),
            'reports_affected_count': len(reports_affected),
            'variants':         _truncate_for_excel(' | '.join(variants)),
            'reports_affected': _truncate_for_excel(' | '.join(reports_affected)),
        }
        if has_expr:
            exprs = grp['Expression'].dropna().astype(str)
            row['canonical_expression'] = _truncate_for_excel(canon_expr)
            sample = sorted(exprs.unique().tolist())[:_EXPR_SAMPLE_COUNT]
            sample = [e[:_EXPR_MAX_CHARS] + ('…' if len(e) > _EXPR_MAX_CHARS else '') for e in sample]
            row['variant_expressions'] = _truncate_for_excel(' || '.join(sample))
            row['distinct_expr_count'] = exprs.nunique()
        rows.append(row)
    return pd.DataFrame(rows)

summary_col  = build_cluster_summary(bucket_column,  'Column',      has_expr=False)
summary_meas = build_cluster_summary(bucket_measure, 'Measure',     has_expr=True)
summary_calc = build_cluster_summary(bucket_calcol,  'Cal. Column', has_expr=True)

# All these are already "candidates" (variant_count >= 2 by construction, since
# we collapsed singletons into '' which is filtered out above).
candidates_col  = summary_col.sort_values('total_occurrences', ascending=False)
candidates_meas = summary_meas.sort_values('total_occurrences', ascending=False)
candidates_calc = summary_calc.sort_values('total_occurrences', ascending=False)

print(f"CONSOLIDATION CANDIDATES (real clusters, singletons excluded):")
print(f"  Column      : {len(candidates_col):,}  (potential merges: {candidates_col['variant_count'].sum() - len(candidates_col):,})")
print(f"  Measure     : {len(candidates_meas):,}  (potential merges: {candidates_meas['variant_count'].sum() - len(candidates_meas):,})")
print(f"  Cal. Column : {len(candidates_calc):,}  (potential merges: {candidates_calc['variant_count'].sum() - len(candidates_calc):,})")


CONSOLIDATION CANDIDATES (real clusters, singletons excluded):
  Column      : 1,397  (potential merges: 2,450)
  Measure     : 938  (potential merges: 2,159)
  Cal. Column : 471  (potential merges: 1,141)


In [32]:
def build_clusters_long(bucket_df: pd.DataFrame, kpi_type: str, has_expr: bool) -> pd.DataFrame:
    """One row per (cluster, variant). Mirrors cred_scoring_engine 'Consolidation Clusters' sheet.

    is_canonical is set on EXACTLY one row per cluster — the most-frequent (name, expr)
    variant tuple (Column ignores expression). The canonical row's variant_name /
    variant_expression IS the canonical KPI / expression — no need to repeat them
    in separate columns.
    """
    real = bucket_df[bucket_df['_cluster_id'] != '']
    rows = []
    for cid, grp in real.groupby('_cluster_id', sort=False):
        canon_name, canon_expr = _pick_canonical_variant(grp, has_expr)

        if has_expr:
            variant_key_col = grp[['KPI/Used Columns', 'Expression']].astype(str).agg(' || '.join, axis=1)
        else:
            variant_key_col = grp['KPI/Used Columns'].astype(str)

        # Iterate each unique variant in the cluster
        for _vkey, vgrp in grp.groupby(variant_key_col, sort=False):
            variant_name = vgrp['KPI/Used Columns'].astype(str).iloc[0]
            variant_expr = vgrp['Expression'].fillna('').astype(str).iloc[0] if has_expr else ''
            variant_reports = sorted(vgrp['Report Name'].dropna().astype(str).unique().tolist())

            if has_expr:
                is_canonical = (variant_name == canon_name) and (variant_expr == canon_expr)
            else:
                is_canonical = (variant_name == canon_name)

            row = {
                'cluster_id': cid,
                'kpi_type': kpi_type,
                'is_canonical': is_canonical,
                'variant_name': variant_name,
            }
            if has_expr:
                row['variant_expression'] = _truncate_for_excel(variant_expr)
            row.update({
                'variant_occurrences': len(vgrp),
                'variant_reports_count': len(variant_reports),
                'variant_reports': _truncate_for_excel(' | '.join(variant_reports)),
            })
            rows.append(row)
    return pd.DataFrame(rows)

long_col  = build_clusters_long(bucket_column,  'Column',      has_expr=False)
long_meas = build_clusters_long(bucket_measure, 'Measure',     has_expr=True)
long_calc = build_clusters_long(bucket_calcol,  'Cal. Column', has_expr=True)

clusters_long = pd.concat([long_col, long_meas, long_calc], ignore_index=True)

# Enforce a tidy column order (variant_expression sits right after variant_name).
_long_col_order = ['cluster_id', 'kpi_type', 'is_canonical', 'variant_name',
                   'variant_expression', 'variant_occurrences',
                   'variant_reports_count', 'variant_reports']
clusters_long = clusters_long.reindex(columns=[c for c in _long_col_order if c in clusters_long.columns])

# Sort: type order (Column, Measure, Cal. Column), then by numeric cluster_id
# ascending (1, 2, 3, ...), then canonical row first within each cluster.
if not clusters_long.empty:
    clusters_long['kpi_type'] = pd.Categorical(
        clusters_long['kpi_type'],
        categories=['Column', 'Measure', 'Cal. Column'],
        ordered=True,
    )
    clusters_long['_sort_num'] = clusters_long['cluster_id'].str.extract(r'-(\d+)').astype(int)
    clusters_long = (
        clusters_long
        .sort_values(['kpi_type', '_sort_num', 'is_canonical'], ascending=[True, True, False])
        .drop(columns='_sort_num')
        .reset_index(drop=True)
    )

# Sanity: every cluster must have exactly 1 canonical row
canon_per_cluster = clusters_long.groupby('cluster_id', observed=True)['is_canonical'].sum()
bad = canon_per_cluster[canon_per_cluster != 1]
print(f"Long-format cluster rows:")
print(f"  Column      : {len(long_col):,} rows")
print(f"  Measure     : {len(long_meas):,} rows")
print(f"  Cal. Column : {len(long_calc):,} rows")
print(f"  COMBINED    : {len(clusters_long):,} rows")
print(f"\nis_canonical rows: {int(clusters_long['is_canonical'].sum()):,}  (should == # clusters = {clusters_long['cluster_id'].nunique():,})")
if len(bad):
    print(f"  ⚠ {len(bad)} clusters have unexpected canonical count: {bad.head().to_dict()}")
else:
    print(f"  ✓ Every cluster has exactly 1 canonical row")
print(f"\nColumns: {clusters_long.columns.tolist()}")


Long-format cluster rows:
  Column      : 3,847 rows
  Measure     : 3,097 rows
  Cal. Column : 1,612 rows
  COMBINED    : 8,556 rows

is_canonical rows: 2,806  (should == # clusters = 2,806)
  ✓ Every cluster has exactly 1 canonical row

Columns: ['cluster_id', 'kpi_type', 'is_canonical', 'variant_name', 'variant_expression', 'variant_occurrences', 'variant_reports_count', 'variant_reports']


In [9]:
# Preview top consolidation candidates per bucket
def _preview(df_cands, cols, n=10):
    if len(df_cands) == 0:
        print("  (no candidates)")
        return
    show = df_cands.head(n)[cols].copy()
    # Truncate long strings for display
    for c in show.select_dtypes(include='object').columns:
        show[c] = show[c].str.slice(0, 80) + show[c].str.len().gt(80).map({True: '…', False: ''})
    print(show.to_string(index=False))

print("=" * 90)
print("TOP 10 COLUMN consolidation candidates")
print("=" * 90)
_preview(candidates_col,
         ['cluster_id', 'canonical_kpi', 'variant_count', 'total_occurrences',
          'reports_affected_count', 'variants'])

print("\n" + "=" * 90)
print("TOP 10 MEASURE consolidation candidates")
print("=" * 90)
_preview(candidates_meas,
         ['cluster_id', 'canonical_kpi', 'variant_count', 'distinct_expr_count',
          'total_occurrences', 'reports_affected_count', 'variants'])

print("\n" + "=" * 90)
print("TOP 10 CAL. COLUMN consolidation candidates")
print("=" * 90)
_preview(candidates_calc,
         ['cluster_id', 'canonical_kpi', 'variant_count', 'distinct_expr_count',
          'total_occurrences', 'reports_affected_count', 'variants'])


TOP 10 COLUMN consolidation candidates
cluster_id                                                                     canonical_kpi  variant_count  total_occurrences  reports_affected_count                                                                          variants
  COL-2671                                                                           Column1             79               1272                     189 Column | Column1 | Column10 | Column101 | Column104 | Column11 | Column12 | Colu…
 COL-13909 Ad Hoc Reporting:Using a 10-point scale where ‘10’ is “Extremely Likely” and ‘0’…             17                544                      32 Ad Hoc Reporting:Using a 10-point scale where ‘10’ is “Extremely Likely” and ‘0’…
   COL-762                                                Account_Planning_Entity_Segment__c              7                539                      85 Account_Planning_EntityId__c | Account_Planning_Entity_Country__c | Account_Plan…
 COL-12118                   

C:\Users\Rishi.Dave\AppData\Local\Temp\ipykernel_10096\1351690895.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in show.select_dtypes(include='object').columns:
C:\Users\Rishi.Dave\AppData\Local\Temp\ipykernel_10096\1351690895.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/u

## Step 6 — Export to Excel

| Sheet | Contents |
|---|---|
| `Overview` | Run settings + per-bucket headline metrics |
| `Consolidation_Column` | Column clusters (≥ 2 variants). `kpi_type` dropped — implied by sheet name. |
| `Consolidation_Measure` | Measure clusters (≥ 2 variants), with canonical/variant expressions |
| `Consolidation_CalCol` | Cal. Column clusters (≥ 2 variants), with canonical/variant expressions |
| `Consolidation Clusters` | **NEW** — long format, one row per (cluster, variant) across all 3 buckets. Mirrors the CRED engine. |
| `Row_Level_With_Cluster` | Every original row + `_cluster_id`. Singletons → `'-1'`. Three always-empty cols dropped. |

The `All_Clusters_*` sheets have been removed (singletons are now collapsed into `'-1'`, so per-bucket "all clusters" is just the consolidation sheet + the singleton bucket — no separate audit view needed).


In [33]:
import os
from datetime import datetime

OUT_DIR = os.path.join(os.path.dirname(os.getcwd()), 'outputs')  # ../outputs
os.makedirs(OUT_DIR, exist_ok=True)
OUT_PATH = os.path.join(OUT_DIR, 'KPI_Consolidation_Output.xlsx')

# Per-bucket "unclustered rows" = rows whose KPI did not match anyone else at the
# threshold. They have an empty cluster_id and do NOT appear in any cluster sheet.
n_col_unclustered  = int((bucket_column ['_cluster_id'] == '').sum())
n_meas_unclustered = int((bucket_measure['_cluster_id'] == '').sum())
n_calc_unclustered = int((bucket_calcol ['_cluster_id'] == '').sum())

# Build Overview sheet
overview = pd.DataFrame([
    {'Metric': 'Run timestamp',                       'Value': datetime.now().strftime('%Y-%m-%d %H:%M:%S')},
    {'Metric': 'Source file',                         'Value': 'Copy of KPI Mapping.xlsx (sheet: Report_KPI Mapping)'},
    {'Metric': 'Similarity threshold (uniform)',      'Value': SIMILARITY_THRESHOLD},
    {'Metric': 'Match rule (Column)',                 'Value': 'fuzz(name_a, name_b) >= threshold'},
    {'Metric': 'Match rule (Measure / CalCol)',       'Value': 'fuzz("name || expr", "name || expr") >= threshold'},
    {'Metric': 'Singleton handling',                  'Value': "KPI didn't match anyone at threshold — left unclustered (empty cluster_id)"},
    {'Metric': 'Cluster ID numbering',                'Value': "Renumbered 1..N per bucket, ordered by impact (rows desc) — COL-1 = biggest Column cluster"},
    {'Metric': '',                                     'Value': ''},
    {'Metric': 'Total input rows',                     'Value': len(df_copy)},
    {'Metric': 'Rows after dropping null Report Name', 'Value': len(df)},
    {'Metric': 'Rows used for clustering',             'Value': len(df_kpi)},
    {'Metric': '',                                     'Value': ''},
    {'Metric': 'Column rows',                          'Value': len(bucket_column)},
    {'Metric': 'Column unique names',                  'Value': bucket_column['_name_norm'].nunique()},
    {'Metric': 'Column real clusters (>=2 variants)',  'Value': len(candidates_col)},
    {'Metric': 'Column unclustered rows (unique KPIs)','Value': n_col_unclustered},
    {'Metric': 'Column potential merges',              'Value': int(candidates_col['variant_count'].sum() - len(candidates_col))},
    {'Metric': '',                                     'Value': ''},
    {'Metric': 'Measure rows',                         'Value': len(bucket_measure)},
    {'Metric': 'Measure unique (name|expr) keys',      'Value': len(meas_keys)},
    {'Metric': 'Measure real clusters (>=2 variants)', 'Value': len(candidates_meas)},
    {'Metric': 'Measure unclustered rows (unique KPIs)','Value': n_meas_unclustered},
    {'Metric': 'Measure potential merges',             'Value': int(candidates_meas['variant_count'].sum() - len(candidates_meas))},
    {'Metric': '',                                     'Value': ''},
    {'Metric': 'Cal.Column rows',                      'Value': len(bucket_calcol)},
    {'Metric': 'Cal.Column unique (name|expr) keys',   'Value': len(calc_keys)},
    {'Metric': 'Cal.Column real clusters (>=2 variants)', 'Value': len(candidates_calc)},
    {'Metric': 'Cal.Column unclustered rows (unique KPIs)', 'Value': n_calc_unclustered},
    {'Metric': 'Cal.Column potential merges',          'Value': int(candidates_calc['variant_count'].sum() - len(candidates_calc))},
])

# Drop kpi_type column from the per-bucket consolidation sheets (redundant with sheet name)
def _strip_kpi_type(df_: pd.DataFrame) -> pd.DataFrame:
    return df_.drop(columns=['kpi_type'], errors='ignore')

cand_col_out  = _strip_kpi_type(candidates_col)
cand_meas_out = _strip_kpi_type(candidates_meas)
cand_calc_out = _strip_kpi_type(candidates_calc)

# Row-level traceability:
#   - drop internal helper cols (originals stay)
#   - drop columns that are 100% empty in the cleaned data:
#     'Repeated', 'Consolidate (Yes/No)', 'Consolidated Report Name' had only
#     1 non-null value each in the full 145K-row source, and 'Table Type' is
#     almost entirely empty too.
_internal_cols  = ['_cluster_raw', '_name_norm', '_expr_norm', '_name_expr_key']
_empty_in_clean = ['Repeated', 'Consolidate (Yes/No)', 'Consolidated Report Name', 'Table Type']
row_level = df_clustered.drop(columns=_internal_cols + _empty_in_clean, errors='ignore').copy()

with pd.ExcelWriter(OUT_PATH, engine='openpyxl') as writer:
    overview.to_excel(writer,              sheet_name='Overview',                 index=False)
    cand_col_out.to_excel(writer,          sheet_name='Consolidation_Column',     index=False)
    cand_meas_out.to_excel(writer,         sheet_name='Consolidation_Measure',    index=False)
    cand_calc_out.to_excel(writer,         sheet_name='Consolidation_CalCol',     index=False)
    clusters_long.to_excel(writer,         sheet_name='Consolidation Clusters',   index=False)
    row_level.to_excel(writer,             sheet_name='Row_Level_With_Cluster',   index=False)

print(f"Wrote: {OUT_PATH}")
print(f"File size: {os.path.getsize(OUT_PATH) / (1024*1024):.1f} MB")


Wrote: c:\Users\Rishi.Dave\OneDrive - Brillio\Desktop\CRED Project\outputs\KPI_Consolidation_Output.xlsx
File size: 9.3 MB
